In [51]:
import tensorflow as tf
from PIL import Image  # Import Pillow for image processing
import numpy as np
import time

class_labels_disease = ["bacterial_leaf_blight", "bacterial_leaf_streak", "bacterial_panicle_blight",
                        "blast", "brown_spot", "dead_heart", "downy_mildew", "hispa", "normal", "tungro"]

class_labels_varieties = ["ADT45", "AndraPonni", "AtchayaPonni",
                          "IR20", "KarnatakaPonni", "Onthanel", "Ponni", "RR", "Surya", "Zonal"]


In [52]:
def preprocess_image(img_path):
    img = Image.open(img_path)
    img = img.resize((224, 224))  # Resize to the target size
    img_array = np.array(img) / 255.0  # Normalize to [0, 1]
    img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
    return img_array

In [53]:
def loader(model_name, model_path):
    print(f"Loading {model_name} model...")
    start_time = time.time()
    model = tf.keras.models.load_model(model_path)
    print(f"{model_name} model loaded in {time.time() - start_time:.2f} seconds.")
    return model

In [54]:
def load_all():
    global task1_model_vgg, task1_model_mobile_net, task1_model_cnn
    global task2_resnet50_model, task2_efficient_net_model, task2_model_cnn
    global task3_nasnet_model, task3_model_cnn

    # Load models
    task1_model_vgg = loader("VGG", "database/models/Task1/task1_vgg_model.h5")
    task1_model_mobile_net = loader("MobileNet", "database/models/Task1/task1_mobile_net_model.h5")
    task1_model_cnn = loader("CNN", "database/models/Task1/task1_cnn_model.h5")

    task2_resnet50_model = loader("Resnet50", "database/models/Task2/task2_resnet50_model.h5")
    task2_efficient_net_model = loader("EfficientNet", "database/models/Task2/task2_efficient_net_model.h5")
    task2_model_cnn = loader("CNN", "database/models/Task2/task2_cnn_model.h5")

    # task3_nasnet_model = loader("Nasnet")
    # task3_model_cnn = loader("CNN")


In [55]:
load_all()

Loading VGG model...
VGG model loaded in 2.18 seconds.
Loading MobileNet model...
MobileNet model loaded in 4.87 seconds.
Loading CNN model...
CNN model loaded in 1.98 seconds.
Loading Resnet50 model...
Resnet50 model loaded in 5.87 seconds.
Loading EfficientNet model...
EfficientNet model loaded in 4.60 seconds.
Loading CNN model...
CNN model loaded in 2.07 seconds.


In [56]:
def classify(file, file_path, mode):
    """
    Classifies the image using preloaded models based on the specified mode: 'disease' or 'variety'.

    Returns:
        dict: A dictionary of predictions from each model if successful.
        None: If an error occurs during processing.
    """
    img_array = preprocess_image(file_path)

    try:
        if mode == "disease":
            # Predict with disease models
            vgg_prediction = task1_model_vgg.predict(img_array)
            mobile_net_prediction = task1_model_mobile_net.predict(img_array)
            cnn_prediction = task1_model_cnn.predict(img_array)

            predictions = {
                "vgg": {
                    "label": class_labels_disease[int(np.argmax(vgg_prediction))],
                    "confidence": f"{float(np.max(vgg_prediction)) * 100:.2f}%",
                },
                "mobile_net": {
                    "label": class_labels_disease[int(np.argmax(mobile_net_prediction))],
                    "confidence": f"{float(np.max(mobile_net_prediction)) * 100:.2f}%",
                },
                "cnn": {
                    "label": class_labels_disease[int(np.argmax(cnn_prediction))],
                    "confidence": f"{float(np.max(cnn_prediction)) * 100:.2f}%",
                },
            }
            return predictions

        elif mode == "variety":
            # Predict with variety models
            resnet50_prediction = task2_resnet50_model.predict(img_array)
            efficient_net_prediction = task2_efficient_net_model.predict(img_array)
            cnn_prediction = task2_model_cnn.predict(img_array)

            predictions = {
                "resnet50": {
                    "label": class_labels_varieties[int(np.argmax(resnet50_prediction))],
                    "confidence": f"{float(np.max(resnet50_prediction)) * 100:.2f}%",
                },
                "efficient_net": {
                    "label": class_labels_varieties[int(np.argmax(efficient_net_prediction))],
                    "confidence": f"{float(np.max(efficient_net_prediction)) * 100:.2f}%",
                },
                "cnn": {
                    "label": class_labels_varieties[int(np.argmax(cnn_prediction))],
                    "confidence": f"{float(np.max(cnn_prediction)) * 100:.2f}%",
                },
            }
            return predictions

        else:
            print(f"Invalid classification mode: {mode}")
            return None

    except Exception as e:
        print(f"Error during classification ({mode}): {e}")
        return None


In [57]:
print(task1_model_vgg)
print(task2_resnet50_model)

In [58]:
import pandas as pd
import os


csv_path = "prediction/prediction_submission.csv"
df = pd.read_csv(csv_path)

for i, row in df.iterrows():
    image_id = row["image_id"]
    image_path = os.path.join("dataset", "test_images", image_id)
    
    try:
        print(f"Classifying image {i+1}/{len(df)}: {image_id}")
        # Run classification
        disease_predictions = classify(image_path, image_path, mode="disease")
        variety_predictions = classify(image_path, image_path, mode="variety")
        
        # Find most confident disease prediction
        label = max(disease_predictions.items(), key=lambda x: float(x[1]['confidence'][:-1]))[1]['label']

        # Find most confident variety prediction
        variety = max(variety_predictions.items(), key=lambda x: float(x[1]['confidence'][:-1]))[1]['label']


        # Update DataFrame
        df.at[i, 'label'] = label
        df.at[i, 'variety'] = variety

    except Exception as e:
        print(f"Failed to classify {image_id}: {e}")
        continue

# Save updated CSV
df.to_csv(csv_path, index=False)


Classifying image 1/3563: 200001.jpg
1/1 [==============================] - 0s 91ms/step
Classifying image 2/3563: 200002.jpg
1/1 [==============================] - 0s 24ms/step
Classifying image 3/3563: 200003.jpg
1/1 [==============================] - 0s 26ms/step
Classifying image 4/3563: 200004.jpg
1/1 [==============================] - 0s 25ms/step
Classifying image 5/3563: 200005.jpg
1/1 [==============================] - 0s 21ms/step
Classifying image 6/3563: 200006.jpg
1/1 [==============================] - 0s 21ms/step
Classifying image 7/3563: 200007.jpg
1/1 [==============================] - 0s 22ms/step
Classifying image 8/3563: 200008.jpg
1/1 [==============================] - 0s 23ms/step
Classifying image 9/3563: 200009.jpg
1/1 [==============================] - 0s 22ms/step
Classifying image 10/3563: 200010.jpg
1/1 [==============================] - 0s 23ms/step
Classifying image 11/3563: 200011.jpg
1/1 [==============================] - 0s 23ms/step
Classifying image 1